<a href="https://colab.research.google.com/github/dharamkkuma/ai-agents-nv-notebooks/blob/main/module-1/01_intro_to_token_factory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 · Notebook 1 of 3 — Introducing Token Factory

**AI Agent Course** · Nebius Token Factory × NVIDIA

Over the three notebooks of this chapter, we build **our first controlled agentic loop**: a model that can request an action, have our Python code execute it, and use the result. This notebook covers the first two steps of that journey: **choosing a model** and **calling it from Python**. Notebook 2 adds the tool loop and Notebook 3 checks whether the loop is fast enough and cheap enough to be usable.

**Prerequisites:** Python 3.10+, basic familiarity with REST APIs. No GPU, no prior LLM experience assumed.

**By the end of this notebook you can:** navigate the Token Factory catalog, read a model card and pick a model for a job, and make chat completion calls from Python — single-turn, styled with system prompts, and multi-turn.

## 1. What is Token Factory?

Open-sourced models are those in which all ingredients - including the weights, documentation, and often research - is publically available. Open source models are particularly powerful because they can be ran on your own servers.

In practice, that takes serious capacity and skills - GPU clusters, serving stacks, scaling, monitoring - and most teams have better things to do. **Managed inference** providers like Nebius Token Factory close that gap: they deploy and operate the open-source models for you. You can interact with these cheaper (and often equally as powerful!) models all in the playground or with simple API calls.

**In this course, we use Token Factory's public models which available to all users and billed per token which makes monitoring cost simple.**

## 2. Setup

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).

<details>
<summary>Show screenshot: Token Factory home</summary>

![Token Factory home](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/token-factory-home.png)

</details>

<details>
<summary>Show screenshot: API Key creation</summary>

![API Key creation](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/api-key-creation.png)

</details>

Now we store that key somewhere our code can find it. We use a `.env` file: a plain text file that holds keys as `NAME=value` lines.
**Create it:**

1. In the same folder as this notebook, create a folder called `lesson` (if it doesn't exist yet).
2. Inside it, create a new plain text file named exactly `.env` (note: no name before the dot!). In VS Code or Jupyter: right-click the `lesson` folder, New File, type `.env`. From a terminal: `touch lesson/.env`.
3. Open it and paste in your key:

```
NEBIUS_API_KEY=paste-your-key-here
```

4. Save. That's it. No quotes, no spaces around the `=`.

**Why a `.env` file at all?** In real development, configuration that changes between people and machines (and *especially* secrets) lives outside your code. The `.env` file holds it; `load_dotenv()` reads it into environment variables. That way the same code runs on your laptop, your teammate's, and the server, and no key ever gets committed. **Key hygiene reminder:** keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it, unless you want someone stealing your Token Factory credits!

**Running in Google Colab?** Colab doesn't have your `.env`. Two options:

- Upload it: open the file browser (folder icon, left side) and drag your `.env` in, then point `load_dotenv()` at it with load_dotenv("path-to-your-env-file").
- Or use **Colab Secrets** (key icon, left side): add `NEBIUS_API_KEY` there, then run `from google.colab import userdata; os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")` instead of `load_dotenv()`.

One common gotcha: your operating system may hide files starting with a dot. If the file seems to vanish after you create it, it's still there. Your code will find it even if Finder or Explorer doesn't show it!

In [1]:
%pip install -q openai python-dotenv rich sympy

In [3]:
import os

from dotenv import load_dotenv
from rich import print

#load_dotenv("lesson/.env")
#%load_ext rich

# Colab users: comment the two lines above and use Colab Secrets instead:
from google.colab import userdata
os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

Keys loaded.

## 3. The model catalog

The model catalog is the home for all models on Token Factory. Here, you can view basic information, and types of availability.

<details>
<summary>Show screenshot: Model catalog</summary>

![Model catalog](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-catalog.png)

</details>

The catalog hosts frontier open models: **GLM 5.2**, **Kimi 3** and the **NVIDIA Nemotron family**, among others.

## 4. The NVIDIA Nemotron ecosystem

Our flagship models for this course are the **NVIDIA Nemotron** series, a family of open-source models from NVIDIA's. The Nemotron series is NVIDIA's latest and most impressive models - spanning both small and efficient to large and frontier-performing.  The family shares training lineage and behavior, so you can prototype on a small one and scale up without rewriting anything. The three we care about:

| | **Nemotron 3 Nano** | **Nemotron 3 Super** | **Nemotron 3 Ultra** |
|---|---|---|---|
| Routing key | `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B` | `nvidia/nemotron-3-super-120b-a12b` | `nvidia/Nemotron-3-Ultra-550b-a55b` |
| Size | Smallest | Medium | Largest |
| Rough character | Fast and very cheap; great for everyday tasks, classification, drafting | The middle ground; strong general model for most workloads | The heavyweight; deepest reasoning, highest price |

A useful mental model: **start small, scale only when the task needs it.** The bigger the model, the more it costs per token and the slower it generates - and for a surprising share of real work, the small one is simply good enough!

**Our workhorse for this chapter is Nemotron 3 Nano.** We'll bring in its bigger siblings when we want a comparison, which will mimmic how you should scale with a real application.

## 5. Reading a model card

If you were deciding on an engine to be put in a car, you would likely read through different spec sheets until you found the one to fit your needs.

Every model in the catalog has a card - this is its spec sheet. The model cards are something you will come back to constantly: for the model string, for prices, for context limits, for what the model can and can't do. Need a fast car? You will pick the strongest - and likely most expensive - option. If you care about efficiency, you would choose one that is smaller and cheaper. The same choices will persist for the models you choose.

Click into **NVIDIA-Nemotron-3-Nano-30B-A3B**, scroll down, and click the model card.

<details>
<summary>Show screenshot: Nemotron 3 Nano model card</summary>

![Nemotron 3 Nano model card](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card.png)

</details>

### Copying the model string

At the top of the endpoint properties you'll find the **routing key** - for our model it's `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`, with a copy icon right next to it. **That exact string is what you pass as `model=` in every API call.** Always copy it from the card rather than typing it; a typo here is the single most common first error, and the API's "model not found" message won't tell you which character you got wrong.

### Reading the model card - what specs are important to us?

**Context window 262k.**
Find this number on the Nano card — it's the ceiling on everything the model can see at once: your prompt, the conversation history, any documents you paste in, *and* the output it generates. They all share this budget. Think of it as the model's working memory. A bigger window means you can hand it an entire codebase or a stack of contracts in one go; a smaller one means you'll have to chop things up and manage what the model gets to see.

**Pricing.**
Pricing is in terms of tokens, and there is a reason for the separation. Input tokens are cheap because the model ingests your whole prompt in a single go; output tokens cost more because the model generates them one at a time. Each one is a full trip through the model. Practical consequence: a chatty model that writes long answers costs you much more (think, what can we do to outputs to reduce this?). For scale: Ultra charges \$1.00 / \$3.00 for the same million tokens — roughly **15× more**.

**Modality: Text-to-text.**
What goes in and what comes out. Our model reads text and writes text, but scroll through the catalog and you'll see vision models (they accept images) and embedding models (they output vectors, not sentences). Checking this field takes two seconds and saves you from building half a project around a model that can't do what you assumed.

**Tool calling / Reasoning: Available.**
We'll return to what these mean later in the course — for now, just note that they're on the card, and that they'll matter a lot once we build agents.

**License.**
Open-weight does not mean do-whatever-you-want. Licenses differ on commercial use, redistribution, and what you can train on the outputs. Important to understand before putting a model in production, or starting work on something you may not be allowed to do.

### Exercise: pick a model by reading its card

Open two cards side by side: **`nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`** and **`nvidia/Nemotron-3-Ultra-550b-a55b`**.

Two jobs. Tor each, decide which model you'd pick, and name the **one or two card fields** that settle it. One sentence of justification each ("it's bigger" doesn't count!):

1. **An everyday assistant** — rephrases emails, drafts short summaries, answers quick questions all day long.
2. **A contract analyst** — answers questions about 300-page legal documents pasted in whole, where a wrong answer is very costly.

*Hint: for one of these, the price gap is important. For the other, look at the context window and ask what a mistake costs.*

## 6. First calls in the UI (Playground)

Before any code, use the **Token Factory playground**:

1. Open the model card for **Nemotron 3 Nano** and click **Go to playground**.
2. Send this prompt:

   > *A survey shows 62% of respondents prefer Product A and 38% prefer Product B, but the group that prefers A is three times smaller in the overall population. Which product actually wins overall? Show your reasoning briefly.*

3. Look at the **metrics below the response** - response time, token counts (it is OK to not fully understand these yet!).
<details>
<summary>Show screenshot: Nemotron response metrics</summary>

![Nemotron response metrics](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/request-metrics.png)

</details>

4. Now run the **exact same prompt** on **Nemotron 3 Ultra** side by side.

**What to compare:** the number of **output tokens** each model spent, how long each took, and the answers themselves - does the big model reason more carefully, or just write more? Did the small one get the right answer anyway? This is your first *evaluation*, and the honest answer to "which model is better" always starts with "for what purpose?".

## 7. The Chat Completions request

The Playground isn't for real applications, it's for *you* to evaluate a model. To use a model in real workflows, you need to call it from code, and the Token Factory API is exactly for that.

A Chat Completions request is the standard way to interact with models via API. It is a structured format for how we communicate to models and how they communicate back. A sample completions request for your model can be seen on the model card:

<details>
<summary>Show screenshot: Chat Completions example</summary>

![Chat Completions example](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card-completions.png)

</details>

### Token Factory's API is OpenAI-compatible

If you ever went through any OpenAI API tutorial, this will be easy-peasy for you: everything works the same way, you just change the `base_url` and the model names. That compatibility is a big deal in practice:

- The entire ecosystem of tools works out of the box
- Drop-in migration from other providers
- Every tutorial on the internet works with a **one-line `base_url` change**

### 7.1 Anatomy of the request

Before the code, the pieces:

- **`model`** *(required)* — the routing key you copied from the card.
- **`messages`** *(required)* — the conversation as a list of roles. **Key note: the API is stateless — you send the whole conversation history every time.** The roles:
  - **`system`** — instructions for how the model should behave. Less about giving it expertise, more about **tone of voice and style**: the same question answered by "a formal analyst" and "a casual intern" reads completely differently. We'll see this live in a minute. Set once, at the top.
  - **`user`** — what the human says. Your prompts go here!
  - **`assistant`** — what the model said. You append its previous replies here so it remembers the conversation.
  - **`tool`** — results from function calls, which we'll meet in the next notebook.
- **Sampling params** *(optional)* — `temperature`, `top_p`, `max_tokens`. We will not worry about these for now.
- **The response object** — what comes back. We'll open one up right after the first call.

### 7.2 Minimal call

The model card provides a ready-made snippet ("Use model in code" - the one below is adapted from it), so you never have to write this boilerplate from memory:

In [5]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

#print(response.choices[0].message.content)
print(response)

ChatCompletion(
    id='chatcmpl-f92fffca-f5a6-4bf9-9283-8588fc2fe0dc',
    choices=[
        Choice(
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content="\nNVIDIA's Nemotron model family targets the generation of high-quality, privacy-compliant
synthetic data to accelerate the training and evaluation of other AI models, particularly when real-world data is 
scarce, sensitive, or difficult to obtain. It enables researchers and developers to create realistic, diverse 
datasets for testing and optimizing AI systems without relying on actual user data.",
                refusal=None,
                role='assistant',
                annotations=None,
                audio=None,
                function_call=None,
                tool_calls=[],
                reasoning='Hmm, the user is asking about NVIDIA\'s Nemotron model family and wants a two-sentence 
answer targeting what it aims to achieve. \n\nFirst, I need to recall what Nemotron is. From my knowledge, Nemotron
is NVIDIA\'s family of models designed specifically for generating synthetic data. The key point is that it\'s used
to create realistic, high-quality data for training other AI models, especially when real data is scarce or 
sensitive.\n\nThe user is part of a market-research team, so they probably care about practical applications. They 
want something concise and impactful—exactly two sentences as specified. I should avoid technical jargon and focus 
on the core value: enabling efficient data creation for AI training.\n\nI should check if there are any common 
misconceptions. Nemotron isn\'t for end-user applications but for backend data synthesis. Also, it\'s part of 
NVIDIA\'s broader AI ecosystem, but the target is very specific: generating data to train other models. \n\nThe 
user\'s deeper need might be understanding how this fits into market research—maybe they\'re looking for tools to 
generate synthetic survey data or test scenarios without real-world constraints. But since they\'re asking for the 
target, I\'ll stick to the technical purpose without overcomplicating it.\n\nNow, crafting two tight sentences: 
First, state what it is (synthetic data generation), second, explain the target (enabling efficient training of 
other AI models). Need to be precise—no fluff. \n\nLet me draft: "NVIDIA\'s Nemotron model family targets the 
generation of high-quality synthetic data to accelerate the training of other AI models, especially when real-world
data is limited or sensitive." Then second sentence: "It enables researchers and developers to create realistic, 
privacy-compliant datasets for testing and training without relying on actual user data."\n\nWait, that\'s two 
sentences? Let me count: First sentence covers the core target, second explains the benefit. Yes, that fits. Make 
sure it\'s exactly two sentences and directly answers the query. \n\nAlso, avoid mentioning "research" since the 
user\'s team is market research, but the model\'s purpose is broader—so I\'ll keep it neutral. \n\nFinal check: Is 
this accurate? Yes, based on NVIDIA\'s official materials. Nemotron is indeed for synthetic data generation to 
train other models. \n\nOkay, response ready: concise, on point, two sentences.\n'
            ),
            stop_reason=None,
            token_ids=None,
            routed_experts=None
        )
    ],
    created=1794080030,
    model='nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B',
    object='chat.completion',
    moderation=None,
    service_tier=None,
    system_fingerprint='vllm-0.22.0-b08c96cd',
    usage=CompletionUsage(
        completion_tokens=576,
        prompt_tokens=44,
        total_tokens=620,
        completion_tokens_details=None,
        prompt_tokens_details=None
    ),
    prompt_logprobs=None,
    prompt_token_ids=None,
    prompt_text=None,
    kv_transfer_params=None
)

### 7.3 What came back? A one-time tour of the response

The response isn't just text - it's a structured object, and a few of its fields matter for everything we do later. Let's dump the whole thing **once**, look around, and then never print it in full again:

In [ ]:
# One-time tour — after this cell, we only ever print .content
print(response.to_json())

# The fields worth remembering:
print("---")
print("content:       ", response.choices[0].message.content[:60], "...")   # the answer
print("finish_reason: ", response.choices[0].finish_reason)                  # why it stopped ("stop" = natural end, "length" = hit max_tokens)
print("usage:         ", response.usage)                                     # tokens in/out — this becomes our COST signal in Notebook 3

Three things to file away: **`content`** is the answer, **`finish_reason`** tells you *why* generation stopped (watch for `"length"` - it means the answer was cut off), and **`usage`** counts prompt and completion tokens - the number your bill is computed from. From here on, our cells print only the content, for readability.

### 7.4 System prompts: same question, two voices

The claim above was that system prompts are mostly about tone and style. Prove it! Same user question, two different personas:

In [6]:
QUESTION = "Is the specialty coffee market growing?"

for persona in [
    "You are a formal senior market analyst. Precise, measured, no exclamation points.",
    "You are an enthusiastic intern on the market-research team. Casual, punchy, emoji welcome.",
]:
    resp = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": QUESTION},
        ],
    )
    print(f"[bold]--- {persona[:40]}... ---[/bold]")
    print(resp.choices[0].message.content, "\n")

--- You are a formal senior market analyst. ... ---

The specialty coffee market has experienced steady expansion over the past decade. Global sales of specialty‑grade 
coffee grew at a compound annual growth rate (CAGR) of approximately 8 % between 2018 and 2023, and projections 
from industry analysts indicate a similar 7‑9 % CAGR through 2028.  

Key drivers of this growth include increasing consumer willingness to pay premium prices for quality, the 
proliferation of coffee‑focused retail concepts, and a broader shift toward premiumization in the food‑and‑beverage
sector. Additionally, sustainability and ethical sourcing have become differentiators that attract a growing 
segment of environmentally conscious shoppers.  

Geographically, the strongest growth rates are observed in North America and Europe, where specialty coffee 
accounts for a larger share of total coffee consumption. Emerging markets in Asia‑Pacific, particularly China and 
South Korea, are also showing accelerated adoption as urbanization and coffee‑culture diffusion expand.  

Challenges to sustained growth include supply‑chain constraints related to climate‑related crop variability, 
volatility in commodity pricing, and the need for continued investment in traceability and certification systems. 
Nonetheless, the overall trajectory remains upward, with specialty coffee expected to capture an increasing 
proportion of total coffee expenditure worldwide.

--- You are an enthusiastic intern on the ma... ---

**Quick answer:** 🌱 **Yes – the specialty coffee market is still on an upward trajectory**, and the momentum shows
no signs of slowing down.  

---

### 📈 What the numbers say  
| Metric | Recent figure (2023‑2024) | Growth rate |
|--------|--------------------------|-------------|
| Global specialty coffee sales | ≈ $45 bn (US + EU + Asia) | **+12 % YoY** |
| US specialty coffee shop count | ~ 38 k locations | **+9 % YoY** |
| Average spend per visit (US) | $5.60 | **+4 % YoY** |

*Sources: Specialty Coffee Association (SCA) 2024 market report, Euromonitor, NCA “Coffee Statistics”. *

---

### 🔎 Why it’s growing (3 quick takeaways)  
1️⃣ **Consumer education** – Millennials & Gen‑Z are digging deeper into origin stories, cupping, and brewing 
methods. The “coffee‑culture” TV shows and Instagram reels are turning coffee into a lifestyle, not just a caffeine
hit. ☕️📱  
2️⃣ **Premiumization** – Shoppers are willing to pay 30‑50 % more for beans labeled “single‑origin,” “shade‑grown,”
or “fair‑trade.” It’s not just a drink; it’s a status cue. 🏆  
3️⃣ **Distribution expansion** – Grocery chains (e.g., Whole Foods, Trader Joe’s) and fast‑casual concepts (e.g., 
Starbucks “Reserve,” Dunkin’ “Brewed”) are adding specialty lines, pushing the segment into mainstream aisles.

---

### 🚀 Actionable insights you can use right now  

| Insight | How to act |
|--------|------------|
| **Target “experience‑first” shoppers** – 68 % say they’ll try a new brand if the packaging tells a story. | 
Design limited‑edition bags with QR codes linking to farm videos. |
| **Leverage online subscriptions** – Subscription boxes grew **+22 %** in 2023. | Offer a “monthly roast” program 
with tiered price points; include brewing guides. |
| **Tap into “home‑brew” kits** – 42 % of coffee drinkers say they bought equipment in the last year. | Bundle 
beans + a small pour‑over set; promote as “DIY café at home.” |
| **Partner with lifestyle influencers** – Micro‑influencers (10k‑50k followers) drive **+18 %** trial lift. | Send
them a starter kit and ask for a “Brew‑with‑me” IG Reel. |
| **Explore emerging markets** – Asia‑Pacific specialty coffee sales rose **+15 %** YoY. | Pilot a small‑batch 
export to Japan’s “third‑wave” cafés. |

---

### 🎯 Bottom line  
The specialty coffee market isn’t just growing; it’s *maturing* into a full‑funnel consumer journey—from farm to 
cup, from digital discovery to in‑store purchase. If you can align your data‑driven storytelling with the 
“origin‑curiosity” that drives these shoppers, you’ve got a sweet spot for rapid brand lift.  

Ready to turn those insights into a conversion‑focused campaign? Let’s brew something together. 🚀💡

Same facts, completely different voice. In an agent, the system prompt is where you pin down *how* it should behave - style, format, boundaries, etc.

### 7.5 Multi-turn: the API is stateless, so *you* carry the memory

Ask a follow-up question in a fresh request and the model has no idea what you're talking about - nothing is stored server-side. "Memory" is literally you resending the conversation. First, fully by hand, so you see every move:

In [7]:
# Turn 1: ask
history = [
    {"role": "system", "content": "You are a concise assistant for a market-research team."},
    {"role": "user", "content": "Name three fast-growing beverage categories."},
]
resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
answer_1 = resp.choices[0].message.content
print(answer_1)

# Turn 2: append the model's answer, then our follow-up, then resend EVERYTHING
history.append({"role": "assistant", "content": answer_1})
history.append({"role": "user", "content": "Which of those is cheapest to enter as a small brand?"})

resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
print("---")
print(resp.choices[0].message.content)   # it knows "those" only because we resent the history

1. **Functional & Botanical Drinks** – beverages fortified with adaptogens, nootropics, vitamins, and herbal 
extracts (e.g., adaptogenic teas, mushroom‑infused waters).  
2. **Low‑/No‑Sugar Plant‑Based Milks & Alternatives** – oat, almond, and mixed‑nut “milks” that are expanding 
beyond traditional dairy‑free niches.  
3. **Hard Seltzers & Low‑ABV Ready‑to‑Drink (RTD) Beverages** – flavored, low‑calorie seltzer‑based drinks with 
added botanical or fruit infusions.  

These categories have shown double‑digit growth rates in recent market reports, driven by consumer demand for 
health‑focused, low‑calorie, and experiential options.

---

**Hard seltzers & low‑ABV RTD beverages** are generally the cheapest to launch as a small brand.  

- **Low upfront CAPEX**: You can start by contract‑manufacturing or co‑packing a seltzer base, which avoids the 
heavy equipment needed for dairy‑based milks or complex botanical extractions.  
- **Simplified formulation**: A seltzer is just carbonated water + flavoring + a modest alcohol or THC/CBD 
addition, reducing R&D and regulatory lab work.  
- **Lower ingredient costs**: Basic flavor extracts, natural sweeteners, and a small amount of alcohol or 
fermentable sugar are inexpensive relative to high‑quality plant‑protein isolates or expensive adaptogenic 
extracts.  
- **Regulatory path**: Alcohol‑based RTDs face fewer health‑claim restrictions than functional botanical drinks, 
and the label requirements are clearer, cutting legal and compliance expenses.  

Thus, among the three categories, hard seltzers/RTDs present the lowest barrier to entry for a small, boot‑strapped
brand.

That works, but the appending pattern is always the same, so wrap it in a tiny loop and you have a chat:

In [8]:
history = [{"role": "system", "content": "You are a concise assistant for a market-research team."}]

def ask(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    return answer

print(ask("Name three fast-growing beverage categories."))
print("---")
print(ask("Which of those is cheapest to enter as a small brand?"))

**Three fast‑growing beverage categories (2023‑2024):**

1. **Non‑Alcoholic & Low‑/No‑Sugar Drinks**  
   - Zero‑sugar sodas, flavored sparkling waters, and low‑calorie mixers are expanding rapidly as consumers cut 
back on sugar and alcohol.

2. **Functional & Wellness Beverages**  
   - Drinks fortified with adaptogens, nootropics, probiotics, electrolytes, or plant‑based protein (e.g., 
kombucha, functional teas, and “better‑for‑you” soda alternatives) are seeing strong double‑digit growth.

3. **Ready‑to‑Drink (RTD) Tea & Coffee‑Based Beverages**  
   - Premium bottled teas, cold‑brew coffee drinks, and ready‑to‑serve coffee cocktails (e.g., cold‑brew with milk 
or tonic) are gaining market share as convenience‑focused consumers seek higher‑quality, on‑the‑go options.

---

**The cheapest category to break into with a small, cash‑strapped brand is usually **  
**1. Non‑Alcoholic & Low‑/No‑Sugar Drinks** – especially flavored sparkling water or zero‑sugar soda alternatives. 

### Why it’s the least‑costly launch point

| Factor | Impact on Cost | How it plays out for a small brand |
|--------|----------------|------------------------------------|
| **Formulation & Ingredients** | Low‑cost raw materials (water, CO₂, natural flavor extracts, a few sweeteners) | 
You can prototype with off‑the‑shelf flavor concentrates and a standard carbonation system. No exotic or regulated 
functional ingredients (e.g., adaptogens) needed. |
| **Regulatory Burden** | Minimal | Most markets only require basic labeling, GRAS‑status for flavorings, and 
compliance with sweetener limits. No health‑claim substantiation or special permits. |
| **Packaging** | Moderate but scalable | Aluminum cans or PET bottles can be sourced in small‑run quantities from 
contract packagers; no glass or complex multi‑layer bottles that require expensive tooling. |
| **Manufacturing** | Simple | A basic **brew‑and‑fill** line can be outsourced. Many co‑packers will fill 
500‑2,000 L batches for a few hundred dollars of setup. |
| **Distribution** | Relatively easy | Direct‑to‑consumer (DTC) e‑commerce, niche grocery chains, and specialty 
retailers all accept small SKUs of sparkling water. No need for a large sales force or liquor‑license negotiations.
|
| **Marketing** | Low to medium | Visual branding (colorful label, minimal ingredient list) is enough; you can 
leverage social media trends, influencer gifting, and “clean‑label” messaging without heavy spend on TV or OOH. |
| **Capital Required** | **$20k‑$60k** (including formulation, small‑batch packaging, initial inventory, and basic 
branding) | This fits the budget of most “startup‑side‑hustle” founders or friends‑and‑family rounds. |

### Comparison with the other two categories

| Category | Typical Cost Drivers | Why it’s more expensive |
|----------|---------------------|------------------------|
| **Functional & Wellness Beverages** | • Need for regulated functional ingredients (e.g., adaptogens, probiotics) 
<br>• Often require higher‑cost formulation expertise and stability testing <br>• May need specific health‑claim 
substantiation and sometimes pre‑market notifications <br>• Packaging may demand special liners or refrigeration | 
R&D, testing, and compliance add **$50k‑$150k+** before any sales. |
| **Ready‑to‑Drink (RTD) Tea & Coffee‑Based Beverages** | • Tea/coffee extraction and brewing equipment <br>• 
Higher‑cost raw materials (premium teas, coffee concentrates) <br>• Often require cold‑fill or aseptic bottling 
lines for shelf‑stable products <br>• May need refrigeration logistics if not shelf‑stable | Startup costs 
typically **$75k‑$200k+**, especially if you target a refrigerated distribution channel. |

### Quick‑Start Checklist (Non‑Alcoholic Sparkling Water)

1. **Pick a simple flavor profile** – e.g., cucumber‑lime, blood‑orange, or berry‑mint.  
2. **Source flavor concentrates** from a B2B flavor house (often sold in 1‑kg packs for <$50).  
3. **Find a co‑packer** that offers “fill‑and‑caps” for carbonated water (many exist at $0.30‑$0.45 per 12‑oz can 
for the first 1,000‑case run).  
4. **Design a label** on a low‑cost design platform (Canva, 99designs) – $200–$500 for 3–5 mock‑ups.  
5. **Set up a DTC storefront** (Shopify + Instagram/Facebook ads) – $30/mo + ad spend.  
6. **Launch a small pilot** (200–500 cases) through local specialty stores or farmers’ markets to test pricing and 
feedback.  

With this lean approach you can start selling in **under 3‑4 months** and keep initial outlay **well under $50k**, 
making it the most cost‑effective entry point among the three categories.

Notice what the loop implies about cost: every turn resends the *whole* history, so long conversations re-pay for everything already said.

### Exercise: try to create a request that does these tasks!

All of these use our market-research assistant. Fill in the `## What goes here?` gaps.

#### 1. Only produce 10 tokens or less per response

In [11]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Give me a one-line headline on the energy-drink market."

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
    max_tokens=10
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)  # what do you expect to see here?

None

---

finish_reason: length

#### 2. Change the *voice*: make the assistant answer like a skeptical veteran analyst who distrusts hype

In [12]:
SYSTEM_PROMPT = """
You are a skeptical veteran market analyst with 25 years of experience.
You distrust hype, exaggerated growth claims, and fashionable narratives.
Challenge assumptions and distinguish facts from speculation.
Prioritise fundamentals, market data, profitability, competitive dynamics,
and long-term sustainability.
Be direct, analytical, and intellectually honest.
Never agree simply to please the user.
If evidence is weak, say so explicitly.
Avoid marketing language and unnecessary optimism.
"""
USER_MESSAGE = "Is the specialty coffee market growing?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

**Short answer:**  
Yes, the specialty‑coffee segment is still expanding, but the pace of growth is slowing and is highly uneven across
regions and channels. Most publicly‑available data show single‑digit compound annual growth rates (CAGR) over the 
last 5‑7 years, with a sharp deceleration after the pandemic‑induced surge of 2020‑2021.

---

## 1. What “specialty coffee” actually means  

| Definition | Typical quality threshold | Typical price premium |
|------------|---------------------------|-----------------------|
| **Specialty Coffee Association (SCA) / Specialty Coffee Association of America (SCAA)** | 80 + points on the 
100‑point cupping scale, traceable origin, specific processing, low defect count | 1.3‑2.5× the price of commodity 
coffee (often $3‑$6 / lb vs. $1‑$1.5 / lb for bulk) |
| **Industry‑wide shorthand** | Focus on “third‑wave” attributes—origin story, brewing method, barista skill | Same
as above; also covers cafés, roasters, and e‑commerce brands that market themselves as “specialty” |

> **Bottom line:** The term covers a *value‑added segment* rather than a distinct commodity class. Growth can be 
driven by price premiums, not just volume.

---

## 2. Global market size & recent CAGR  

| Source (year) | Reported 2023 market size* | Reported CAGR (2020‑2025) | Notes |
|--------------|----------------------------|---------------------------|-------|
| **Grand View Research** – “Specialty Coffee Market Size, Share & Trends” (2024) | **≈ $45 bn** (incl. beans & 
retail) | **7.5 %** (2020‑2025) | Includes “third‑wave cafés”, retail beans, and subscription services. |
| **Statista** – “Specialty coffee market size worldwide” (2023) | **≈ $42 bn** | **6‑7 %** (2021‑2026) | Uses 
“specialty coffee” as a consumer‑behaviour segment; excludes traditional commodity coffee. |
| **Mintel** – “Coffee and Coffee‑Based Drinks – Global Overview” (2022) | **≈ $38 bn** (beans only) | **5.5 %** 
(2020‑2025) | Based on retail sales; diverges from larger “total coffee” market (~$150 bn). |
| **Euromonitor** – “Coffee in the Global Market” (2023) | **≈ $122 bn** (total coffee) – specialty coffee accounts
for **~34 %** of that value | **4‑5 %** (2022‑2027) | Provides the most conservative estimate; specialty‑segment 
growth is embedded within the broader coffee market. |

\*Figures are expressed in **USD** and rounded to the nearest billion. They typically combine **bean sales** 
(retail & food‑service) and **café‑level revenue**, depending on the provider.

### Trend summary  

| Year | Global specialty‑coffee revenue (est.) | YoY growth | Driver |
|------|----------------------------------------|-----------|--------|
| 2018 | $30 bn | — | Early “third‑wave” expansion |
| 2019 | $32 bn | +6.7 % | Continued premiumization |
| 2020 | $34 bn | +6.3 % | Pandemic‑home‑brew surge |
| 2021 | $38 bn | +11.8 % | Coffee‑shop reopenings, subscription boom |
| 2022 | $41 bn | +8.2 % | Inflation‑adjusted price hikes |
| 2023 | $45 bn | +9.8 % | Premium‑segment “experience” spend |
| 2024 (proj.) | $49 bn | +8.9 % | Maturation of subscription & ready‑to‑drink (RTD) formats |
| 2025 (proj.) | $53 bn | +8.2 % | Slow‑down as market saturates in key Western markets |

**Takeaway:** Early pandemic years delivered a **double‑digit jump**, but post‑2021 growth has settled into the 
**6‑9 % range**. The “high‑growth” narrative of the past 3‑4 years is therefore **moderately overstated** when 
extrapolated beyond 2023.

---

## 3. Geographic breakdown (2023)

| Region | Share of global specialty‑coffee sales | Recent growth rate |
|--------|----------------------------------------|--------------------|
| **North America** (U.S. + Canada) | **≈ 38 %** | 7‑9 % YoY (U.S. retail beans) |
| **Europe** (EU‑27 + UK) | **≈ 30 %** | 5‑7 % YoY (driven by Scandinavia, Germany, UK) |
| **Asia‑Pacific** (China, Japan, Korea, Australia) | **≈ 18 %** | 12‑15 % YoY (China’s “new‑age” coffee culture) |
| **Latin America** (origins + imports) | **≈ 9 %** | 4‑6 % 

#### 3. Challenge! Make the model produce 10 identical responses with:

```python
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."
```

**Hint: you may need to use a loop and a certain parameter that makes the model more deterministic (less creative)**

In [13]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
   n=10,
   temperature=0
)

for i, choice in enumerate(response.choices, 1):
    print(f"\n--- Response {i} ---")
    print(choice.message.content)

BadRequestError: Error code: 400 - {'error': {'type': 'invalid_request_error', 'message': "Parameter 'n' must be less than or equal to 3 for non-dedicated models", 'param': 'n', 'code': None}, 'detail': "Invalid request. Please check the parameters and try again. Details: Parameter 'n' must be less than or equal to 3 for non-dedicated models"}

## Wrap-up!

You created an account and an API key, and stored it in a dot-env file so it stays out of your code. You learned to read a model card - the routing key, the context window and the three things sharing it, the two pricing rates and why output costs more, modality, and the license. You tested a model in the playground and compared a small one against a large one on the same prompt. And then you built the same interaction in code: a single request, the response object with its content, finish reason, and usage, a system prompt to control voice, and a multi-turn conversation where you carry the history yourself.

Together, that means you can choose a model for a job, call it from Python, and hold a real conversation with it while knowing exactly what each request cost you.

You now have a working foundation - a model you've chosen deliberately, and the code to talk to it.
